# Pendeteksi Cuaca (Colab)
1. **Runtime → Change runtime type → T4 GPU**
2. Upload `data.zip` ke Google Drive (My Drive). Isinya `data/<kelas>/*.jpg`, `data.zip` 14 kelas, tapi yang dipakai cuma 7 (lihat `KELAS` di cell 3).
3. Jalankan cell dari atas ke bawah.

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')
!unzip -qo /content/drive/MyDrive/data.zip -d /content
!ls /content/data

In [ ]:
import json, keras, numpy as np

SIZE = (224, 224)
EPOCHS = 20  # ganti di sini

# Cuma kelas yang relevan di Indonesia (snow, frost, rime, glaze, sandstorm, dew, hail diabaikan)
KELAS = ['cloudy', 'fogsmog', 'lightning', 'rain', 'rainbow', 'shine', 'sunrise']
kw = dict(validation_split=0.2, seed=42, image_size=SIZE, batch_size=32, class_names=KELAS)
train_ds = keras.utils.image_dataset_from_directory('/content/data', subset='training', **kw)
val_ds = keras.utils.image_dataset_from_directory('/content/data', subset='validation', **kw)
classes = train_ds.class_names
print(classes)

In [ ]:
base = keras.applications.MobileNetV2(input_shape=SIZE + (3,), include_top=False, weights='imagenet')
base.trainable = False  # ponytail: backbone frozen; unfreeze top layers + low LR if accuracy plateaus
model = keras.Sequential([
    keras.Input(SIZE + (3,)),
    keras.layers.RandomFlip('horizontal'),
    keras.layers.RandomRotation(0.1),
    keras.layers.Rescaling(1 / 127.5, offset=-1),  # MobileNetV2 expects [-1, 1]
    base,
    keras.layers.GlobalAveragePooling2D(),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(len(classes), activation='softmax'),
])
model.compile('adam', 'sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS,
          callbacks=[keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)])

In [ ]:
# Tes: upload foto, lihat prediksinya
for name in files.upload():
    img = keras.utils.img_to_array(keras.utils.load_img(name, target_size=SIZE))
    probs = model.predict(img[None], verbose=0)[0]
    print(name)
    for i in np.argsort(probs)[::-1]:
        print(f'{classes[i]:>10}: {probs[i]:.1%}')

In [ ]:
# Simpan & download (buat dipakai di laptop: python cuaca.py predict foto.jpg)
model.save('model_cuaca.keras')
json.dump(classes, open('labels.json', 'w'))
files.download('model_cuaca.keras')
files.download('labels.json')